In [ ]:
def plot_features_num_regression(df, target_col="", columns=None, umbral_corr=0, pvalue_entrada=None):
    """
    Objetivo:
    - Filtra columnas numéricas según su correlación (Pearson) con la variable target (regresión)
      y pinta pairplot(s) con target_col + variables seleccionadas.

    Comportamiento:
    - Si 'columns' está vacía (None o []), se usan todas las variables numéricas del dataframe (excepto target_col).
    - Si 'columns' tiene valores, solo se evalúan esas columnas (ignorando las que no existan o no sean numéricas).
    - Se seleccionan solo las que cumplan:
        abs(corr) > umbral_corr
      y si pvalue_entrada no es None:
        pvalue_calculado < pvalue_entrada
      (esto equivale a significación estadística al nivel 1 - pvalue_entrada).

    EXTRA:
    - Si hay muchas columnas seleccionadas, se pintan varios pairplots con un máximo de 5 columnas por gráfico
      (siendo siempre una de ellas target_col).

    Argumentos:
    df (pd.DataFrame): DataFrame con las variables.
    target_col (str): Columna target (debe ser numérica).
    columns (list[str] o None): Lista de columnas candidatas. Si None o [], se usan todas las numéricas.
    umbral_corr (float): Umbral de correlación entre 0 y 1 (por defecto 0).
    pvalue_entrada (float o None): Umbral de p-valor (si None, no se filtra por significación).

    Retorna:
    list[str] o None: Lista de columnas seleccionadas (sin incluir target_col).
                      Si los argumentos no son válidos retorna None e imprime el motivo.
    """
    from scipy.stats import pearsonr
    import pandas as pd
    import numpy as np
    import seaborn as sns
    import matplotlib.pyplot as plt

    # --------------------------
    # Checks de entrada (como en función 3)
    # --------------------------
    if not isinstance(df, pd.DataFrame):
        print("El argumento df debe ser un pandas DataFrame.")
        return None

    if not isinstance(target_col, str) or target_col.strip() == "":
        print("target_col debe ser un string no vacío.")
        return None

    if target_col not in df.columns:
        print("La columna target no se encuentra en el DataFrame.")
        return None

    if not pd.api.types.is_numeric_dtype(df[target_col]):
        print("La columna target debe ser numérica (continua o discreta de alta cardinalidad) para regresión.")
        return None

    if not isinstance(umbral_corr, (int, float)) or not (0 <= float(umbral_corr) <= 1):
        print("El umbral de correlación debe ser un número entre 0 y 1.")
        return None
    umbral_corr = float(umbral_corr)

    if pvalue_entrada is not None:
        if not isinstance(pvalue_entrada, (int, float)) or not (0 < float(pvalue_entrada) <= 1):
            print("pvalue_entrada debe ser None o un número en (0, 1].")
            return None
        pvalue_entrada = float(pvalue_entrada)

    # --------------------------
    # Preparar columnas candidatas
    # --------------------------
    if columns is None or columns == []:
        # Todas las numéricas menos el target
        columnas_candidatas = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
        if target_col in columnas_candidatas:
            columnas_candidatas.remove(target_col)
    else:
        # Validar que sea lista de strings
        if not isinstance(columns, list) or not all(isinstance(c, str) for c in columns):
            print("columns debe ser una lista de strings, una lista vacía o None.")
            return None

        # Filtrar: que existan y sean numéricas (e ignorar target)
        columnas_candidatas = []
        for c in columns:
            if c not in df.columns:
                print(f"Aviso: la columna '{c}' no existe y se ignora.")
                continue
            if c == target_col:
                continue
            if pd.api.types.is_numeric_dtype(df[c]):
                columnas_candidatas.append(c)
            else:
                print(f"Aviso: la columna '{c}' no es numérica y se ignora.")

    if len(columnas_candidatas) == 0:
        print("Aviso: No hay columnas numéricas candidatas para evaluar/pintar.")
        return []

    # --------------------------
    # Filtrar por correlación (+ pvalue si aplica)
    # --------------------------
    seleccionadas = []

    for col in columnas_candidatas:
        # Dropna por pareja para evitar errores de pearsonr
        tmp = df[[target_col, col]].dropna()
        if tmp.shape[0] < 2:
            continue

        # Si una variable es constante, pearsonr falla
        if tmp[target_col].nunique() < 2 or tmp[col].nunique() < 2:
            continue

        corr, pval = pearsonr(tmp[col], tmp[target_col])

        if abs(corr) > umbral_corr:
            if pvalue_entrada is None:
                seleccionadas.append(col)
            else:
                if pval < pvalue_entrada:
                    seleccionadas.append(col)

    if len(seleccionadas) == 0:
        print("Aviso: Ninguna columna cumple el umbral de correlación (y pvalue si aplica).")
        return []

    # --------------------------
    # Pintar pairplots en chunks: target + 4 columnas => 5 columnas máximo
    # --------------------------
    def chunk_list(lst, size=4):
        return [lst[i:i+size] for i in range(0, len(lst), size)]

    chunks = chunk_list(seleccionadas, size=4)

    for i, chunk in enumerate(chunks, start=1):
        cols_plot = [target_col] + chunk
        data_plot = df[cols_plot].dropna()

        if data_plot.shape[0] < 2:
            print(f"Aviso: No hay suficientes datos para pintar el chunk {i}.")
            continue

        g = sns.pairplot(data_plot, corner=True, diag_kind="hist")
        g.fig.suptitle(f"Pairplot numérico {i}/{len(chunks)} - target: {target_col}", y=1.02)
        plt.show()

    return seleccionadas
